# 실습 3: MLP 모델 구축 및 훈련 파이프라인 완성 (The Synthesis)

**목표:** 지금까지 배운 모든 개념(Tensor, Autograd, nn.Module, Loss, Optimizer)을 **하나의 완결된 훈련 파이프라인**으로 통합하여 모델을 학습시킨다.

## 개념 복기 및 이론 점검
1. **모듈화의 중요성:** `nn.Module`을 상속받아 모델을 **클래스**로 관리 → 재사용/확장이 쉬움.
2. **훈련 루프의 4단계 반복:**
   1. 데이터 로드
   2. 순전파(Forward)
   3. 역전파(Backward)
   4. 가중치 업데이트(Optimizer Step)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/2주차/lab_03_mlp_training_loop.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(0)

# device 설정
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

---
## 1. MLP 모델 클래스 정의

구조: **입력층(2) → 은닉층1(16) → 은닉층2(16) → 출력층(3)**  
각 은닉층 뒤에 `ReLU` 활성화 함수 적용 → 비선형성 확보.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, out_dim=3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)         # 분류용: 마지막에는 활성화 없이 logits 반환
        return x


model = MLP().to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n학습 대상 파라미터 개수: {n_params}")

---
## 2. 가상 데이터셋 준비 (3개 클래스 분류)

2D 평면 위에서 3개의 중심점 주변으로 군집된 점들을 분류하는 문제를 만듭니다.

In [ ]:
def make_dataset(n_per_class=200):
    centers = torch.tensor([[0.0, 0.0], [3.0, 3.0], [-3.0, 3.0]])
    xs, ys = [], []
    for cls_idx, c in enumerate(centers):
        pts = c + 0.8 * torch.randn(n_per_class, 2)
        xs.append(pts)
        ys.append(torch.full((n_per_class,), cls_idx, dtype=torch.long))
    X = torch.cat(xs, dim=0)
    y = torch.cat(ys, dim=0)
    perm = torch.randperm(len(X))
    return X[perm], y[perm]


X_train, y_train = make_dataset(200)
X_test,  y_test  = make_dataset(100)
print(f"훈련 데이터: X={X_train.shape}, y={y_train.shape}")
print(f"테스트 데이터: X={X_test.shape}, y={y_test.shape}")

# 데이터 시각화
plt.figure(figsize=(5, 5))
for c in range(3):
    mask = y_train == c
    plt.scatter(X_train[mask, 0], X_train[mask, 1], label=f"class {c}", alpha=0.6)
plt.legend(); plt.title("Training Data"); plt.grid(True); plt.show()

In [ ]:
# DataLoader 사용 — 미니배치로 나눠서 공급
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# 한 배치 꺼내보기
xb, yb = next(iter(train_loader))
print(f"배치 X shape: {xb.shape}, 배치 y shape: {yb.shape}")

---
## 3. 훈련 루프 구축 (핵심)

각 epoch 안에서 배치마다 **4단계**를 순서대로 실행합니다:
1. **Forward** — 예측값(Output) 얻기
2. **Loss** — 예측과 정답 비교
3. **Backward** — `loss.backward()` 로 기울기 계산
4. **Optimization** — `optimizer.step()` 으로 가중치 업데이트

In [ ]:
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()                    # 분류용 손실 함수
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Adam Optimizer

epochs = 50
train_loss_hist = []
train_acc_hist = []

for epoch in range(epochs):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        # ───── 단계 1: Forward ─────
        logits = model(xb)

        # ───── 단계 2: Loss ─────
        loss = criterion(logits, yb)

        # ───── 단계 3: Backward ─────
        optimizer.zero_grad()
        loss.backward()

        # ───── 단계 4: Optimization ─────
        optimizer.step()

        # 배치 통계
        epoch_loss += loss.item() * xb.size(0)
        epoch_correct += (logits.argmax(dim=1) == yb).sum().item()
        epoch_total += xb.size(0)

    avg_loss = epoch_loss / epoch_total
    avg_acc = epoch_correct / epoch_total
    train_loss_hist.append(avg_loss)
    train_acc_hist.append(avg_acc)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | Loss = {avg_loss:.4f} | Acc = {avg_acc*100:.2f}%")

---
## 4. 결과 측정 및 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_loss_hist)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True)

axes[1].plot(train_acc_hist, color="green")
axes[1].set_title("Training Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# 테스트 세트로 평가
model.eval()
with torch.no_grad():
    X_test_d = X_test.to(device)
    y_test_d = y_test.to(device)
    logits = model(X_test_d)
    preds = logits.argmax(dim=1)
    test_acc = (preds == y_test_d).float().mean().item()
print(f"📊 테스트 정확도: {test_acc*100:.2f}%")

In [ ]:
# 결정 경계(Decision Boundary) 시각화
xx, yy = torch.meshgrid(
    torch.linspace(-6, 6, 200),
    torch.linspace(-4, 6, 200),
    indexing="xy",
)
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)

model.eval()
with torch.no_grad():
    zz = model(grid).argmax(dim=1).cpu().reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx.numpy(), yy.numpy(), zz.numpy(), alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5])
for c in range(3):
    mask = y_train == c
    plt.scatter(X_train[mask, 0], X_train[mask, 1], label=f"class {c}", edgecolors="k", s=20)
plt.legend(); plt.title("Learned Decision Boundary"); plt.show()

---
## ✅ 학습 결과 정리 (Verification)

- Loss 값이 epoch이 지남에 따라 꾸준히 감소하는 것을 **직접 목격**했다.
- 모델의 정확도가 상승하고, 실제 데이터 분포에 맞는 **결정 경계**가 학습되었다.
- 테스트 데이터에서도 높은 정확도를 달성했다 (일반화 성공).

### 🎯 핵심 결론

딥러닝 모델 개발은 두 가지가 **분리**되어 있다:

| 역할 | 예시 코드 |
|------|-----------|
| **모델 구조 정의** | `class MLP(nn.Module): ...` |
| **훈련 파이프라인 관리** | Forward → Loss → Backward → Step |

이 구조 덕분에 **같은 훈련 루프**로 모델만 바꿔 가며(MLP → CNN → Transformer) 재사용할 수 있다.  
→ 이것이 PyTorch의 설계 철학이자 실무에서의 강력함이다.